In [0]:
from pyspark.sql import functions as F

# Read the raw session-level Bronze table
bronze_df = spark.table("ledgr.bronze.sessions_raw")

# Explode the spans array so each row becomes one CALL, not one session
exploded_df = bronze_df.select(
    "session_id",
    "run_id",
    "harness",
    "benchmark",
    "benchmark_subset",
    "success",
    "agent_cost",
    "execution_time",
    F.posexplode("spans").alias("span_index", "span")
)

print(f"Session-level rows (Bronze): {bronze_df.count()}")
print(f"Call-level rows (after explode): {exploded_df.count()}")

In [0]:
# Extract fields from the nested span struct into proper columns
normalized_df = exploded_df.select(
    "session_id",
    "run_id",
    "harness",
    "benchmark",
    "success",
    "agent_cost",
    "execution_time",
    F.col("span.span_id").alias("call_id"),
    F.col("span.trace_id").alias("trace_id"),
    F.col("span.parent_span_id").alias("parent_span_id"),
    F.col("span.start_time").alias("start_time"),
    F.col("span.end_time").alias("end_time"),
    F.col("span.status.code").alias("status_code"),
    F.col("span.status.message").alias("status_message"),
    F.col("span.attributes.`gen_ai.request.model`").alias("model_request"),
    F.col("span.attributes.`gen_ai.response.model`").alias("model_response"),
    F.col("span.attributes.`gen_ai.usage.input_tokens`").alias("input_tokens"),
    F.col("span.attributes.`gen_ai.usage.output_tokens`").alias("output_tokens"),
    F.col("span.attributes.`gen_ai.provider.name`").alias("provider"),
    F.length(F.col("span.attributes.`gen_ai.input.messages`")).alias("input_message_length"),
    F.length(F.col("span.attributes.`gen_ai.output.messages`")).alias("output_message_length"),
    F.col("span.attributes.`gen_ai.tool.definitions`").isNotNull().alias("has_tool_definitions"),
)

# task_id = session_id, matching the original execution hierarchy design
normalized_df = normalized_df.withColumn("task_id", F.col("session_id"))

print(f"Normalized rows: {normalized_df.count()}")
normalized_df.printSchema()
display(normalized_df.limit(5))

In [0]:
# Check exactly which session is missing, and whether it's consistent
raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/raw/")
print(f"Row count: {raw_df.count()}")

# Check for null session_ids (a common cause of a silently dropped row)
null_session_count = raw_df.filter(F.col("session_id").isNull()).count()
print(f"Rows with null session_id: {null_session_count}")

# Check for exact duplicate session_ids (would cause undercounting if any collapse)
distinct_sessions = raw_df.select("session_id").distinct().count()
print(f"Distinct session_ids: {distinct_sessions}")
print(f"Total rows: {raw_df.count()}")
print(f"Difference (duplicates if >0): {raw_df.count() - distinct_sessions}")

In [0]:
harness_rates = {
    "claude_code": 0.2928,
    "openai_solo": 0.026,
    "smolagents_code": 0.0005,
    "tool_calling": 0.0023,
    "tool_calling_with_shortlisting": 0.0006,
}
model_rates = {
    "DeepSeek-V3.2": 0.0574,
    "Kimi-K2.5": 0.0707,
    "claude-opus-4-5": 0.2294,
    "gemini-3-pro-preview": 0.0235,
    "gpt-5.2-2025-12-11": 0.0,
}
mean_model_rate = sum(model_rates.values()) / len(model_rates)
SEED = 42

silver_df = spark.table("ledgr.silver.calls_normalized")

harness_map = F.create_map([F.lit(x) for pair in harness_rates.items() for x in pair])
model_map = F.create_map([F.lit(x) for pair in model_rates.items() for x in pair])

df = (silver_df
    .withColumn("attempt_id", F.concat(F.col("call_id"), F.lit("_attempt_1")))
    .withColumn("harness_rate", harness_map[F.col("harness")])
    .withColumn("model_rate", model_map[F.col("model_request")])
    .withColumn("relative_risk_raw", F.col("model_rate") / F.lit(mean_model_rate))
    .withColumn("relative_risk", F.least(F.greatest(F.col("relative_risk_raw"), F.lit(0.5)), F.lit(2.0)))
    .withColumn("injection_probability", F.least(F.col("harness_rate") * F.col("relative_risk"), F.lit(0.95)))
    .withColumn("hash_uniform", (F.pmod(F.xxhash64(F.col("call_id"), F.lit(SEED)), F.lit(1000000)) / F.lit(1000000.0)))
    .withColumn("is_selected_for_injection", F.col("hash_uniform") < F.col("injection_probability"))
)

selected_count = df.filter(F.col("is_selected_for_injection")).count()
total_count = df.count()
print(f"Total calls: {total_count}")
print(f"Selected for synthetic retry injection: {selected_count}")
print(f"Injection rate: {selected_count/total_count:.4f}")
print(f"Expected (local pipeline): 23618 / 241473 = {23618/241473:.4f}")

In [0]:
normalized_df.write.format("delta").mode("overwrite").saveAsTable("ledgr.silver.calls_normalized")
print(f"Rows written: {spark.table('ledgr.silver.calls_normalized').count()}")

In [0]:
print("Per-harness comparison:")
df.groupBy("harness").agg(
    F.count("*").alias("n"),
    F.avg(F.col("is_selected_for_injection").cast("int")).alias("actual_rate"),
    F.first("harness_rate").alias("target_harness_rate")
).orderBy("harness").show(truncate=False)

print("Per-model comparison:")
df.groupBy("model_request").agg(
    F.count("*").alias("n"),
    F.avg(F.col("is_selected_for_injection").cast("int")).alias("actual_rate"),
    F.first("relative_risk").alias("relative_risk")
).orderBy("model_request").show(truncate=False)

In [0]:
# Fail-loud check: confirm every harness and model in the data is mapped in config
actual_harnesses = set(row.harness for row in silver_df.select("harness").distinct().collect())
actual_models = set(row.model_request for row in silver_df.select("model_request").distinct().collect())

unmapped_harnesses = actual_harnesses - set(harness_rates.keys())
unmapped_models = actual_models - set(model_rates.keys())

if unmapped_harnesses:
    raise ValueError(f"harness_rates config is missing rates for: {unmapped_harnesses}")
if unmapped_models:
    raise ValueError(f"model_rates config is missing rates for: {unmapped_models}")

print("All harnesses and models confirmed mapped in config. Safe to proceed.")

In [0]:
from pyspark.sql import functions as F, SparkSession

HARNESS_RATES = {
    "claude_code": 0.2928,
    "openai_solo": 0.026,
    "smolagents_code": 0.0005,
    "tool_calling": 0.0023,
    "tool_calling_with_shortlisting": 0.0006,
}
MODEL_RATES = {
    "DeepSeek-V3.2": 0.0574,
    "Kimi-K2.5": 0.0707,
    "claude-opus-4-5": 0.2294,
    "gemini-3-pro-preview": 0.0235,
    "gpt-5.2-2025-12-11": 0.0,
}
MEAN_MODEL_RATE = sum(MODEL_RATES.values()) / len(MODEL_RATES)
SEED = 42


def explode_bronze_sessions(bronze_df):
    """Explode session-level Bronze data into call-level rows."""
    return bronze_df.select(
        "session_id", "run_id", "harness", "benchmark", "benchmark_subset",
        "success", "agent_cost", "execution_time",
        F.posexplode("spans").alias("span_index", "span")
    )


def extract_call_fields(exploded_df):
    """Extract gen_ai.* fields from the nested span struct into flat columns."""
    df = exploded_df.select(
        "session_id", "run_id", "harness", "benchmark", "success",
        "agent_cost", "execution_time",
        F.col("span.span_id").alias("call_id"),
        F.col("span.trace_id").alias("trace_id"),
        F.col("span.parent_span_id").alias("parent_span_id"),
        F.col("span.start_time").alias("start_time"),
        F.col("span.end_time").alias("end_time"),
        F.col("span.status.code").alias("status_code"),
        F.col("span.status.message").alias("status_message"),
        F.col("span.attributes.`gen_ai.request.model`").alias("model_request"),
        F.col("span.attributes.`gen_ai.response.model`").alias("model_response"),
        F.col("span.attributes.`gen_ai.usage.input_tokens`").alias("input_tokens"),
        F.col("span.attributes.`gen_ai.usage.output_tokens`").alias("output_tokens"),
        F.col("span.attributes.`gen_ai.provider.name`").alias("provider"),
        F.length(F.col("span.attributes.`gen_ai.input.messages`")).alias("input_message_length"),
        F.length(F.col("span.attributes.`gen_ai.output.messages`")).alias("output_message_length"),
        F.col("span.attributes.`gen_ai.tool.definitions`").isNotNull().alias("has_tool_definitions"),
    )
    return df.withColumn("task_id", F.col("session_id"))


def validate_config_coverage(df, harness_rates=HARNESS_RATES, model_rates=MODEL_RATES):
    """Fail-loud check: every harness/model in the data must have a configured rate."""
    actual_harnesses = set(row.harness for row in df.select("harness").distinct().collect())
    actual_models = set(row.model_request for row in df.select("model_request").distinct().collect())

    unmapped_harnesses = actual_harnesses - set(harness_rates.keys())
    unmapped_models = actual_models - set(model_rates.keys())

    if unmapped_harnesses:
        raise ValueError(f"harness_rates config is missing rates for: {unmapped_harnesses}")
    if unmapped_models:
        raise ValueError(f"model_rates config is missing rates for: {unmapped_models}")
    return True


def compute_injection_probability(df, harness_rates=HARNESS_RATES, model_rates=MODEL_RATES,
                                     mean_model_rate=MEAN_MODEL_RATE, seed=SEED):
    """Compute per-row synthetic retry injection probability, harness-anchored with clipped model relative risk."""
    harness_map = F.create_map([F.lit(x) for pair in harness_rates.items() for x in pair])
    model_map = F.create_map([F.lit(x) for pair in model_rates.items() for x in pair])

    return (df
        .withColumn("attempt_id", F.concat(F.col("call_id"), F.lit("_attempt_1")))
        .withColumn("harness_rate", harness_map[F.col("harness")])
        .withColumn("model_rate", model_map[F.col("model_request")])
        .withColumn("relative_risk_raw", F.col("model_rate") / F.lit(mean_model_rate))
        .withColumn("relative_risk", F.least(F.greatest(F.col("relative_risk_raw"), F.lit(0.5)), F.lit(2.0)))
        .withColumn("injection_probability", F.least(F.col("harness_rate") * F.col("relative_risk"), F.lit(0.95)))
        .withColumn("hash_uniform", (F.pmod(F.xxhash64(F.col("call_id"), F.lit(seed)), F.lit(1000000)) / F.lit(1000000.0)))
        .withColumn("is_selected_for_injection", F.col("hash_uniform") < F.col("injection_probability"))
    )

In [0]:
import sys
print(sys.path)

import os
print(os.getcwd())
print(os.listdir("."))

In [0]:
import pytest
from pyspark.sql import SparkSession, Row
from databricks.silver_transform import (
    validate_config_coverage,
    compute_injection_probability,
    HARNESS_RATES,
    MODEL_RATES,
)


@pytest.fixture(scope="module")
def spark():
    return SparkSession.builder.appName("ledgr-tests").getOrCreate()


def test_validate_config_coverage_passes_for_known_values(spark):
    df = spark.createDataFrame([
        Row(harness="claude_code", model_request="DeepSeek-V3.2"),
    ])
    assert validate_config_coverage(df) is True


def test_validate_config_coverage_raises_for_unmapped_harness(spark):
    df = spark.createDataFrame([
        Row(harness="totally_unknown_harness", model_request="DeepSeek-V3.2"),
    ])
    with pytest.raises(ValueError, match="harness_rates"):
        validate_config_coverage(df)


def test_validate_config_coverage_raises_for_unmapped_model(spark):
    df = spark.createDataFrame([
        Row(harness="claude_code", model_request="totally_unknown_model"),
    ])
    with pytest.raises(ValueError, match="model_rates"):
        validate_config_coverage(df)


def test_injection_probability_relative_risk_upper_clip(spark):
    # claude-opus-4-5 has the highest model rate, should clip relative_risk to 2.0
    df = spark.createDataFrame([
        Row(call_id="c1", harness="claude_code", model_request="claude-opus-4-5"),
    ])
    result = compute_injection_probability(df).collect()[0]
    assert result.relative_risk == 2.0


def test_injection_probability_relative_risk_lower_clip(spark):
    # gpt-5.2 has a 0.0 model rate, should clip relative_risk to 0.5, not 0.0
    df = spark.createDataFrame([
        Row(call_id="c1", harness="claude_code", model_request="gpt-5.2-2025-12-11"),
    ])
    result = compute_injection_probability(df).collect()[0]
    assert result.relative_risk == 0.5


def test_injection_probability_is_deterministic(spark):
    # Same call_id + seed should always produce the same hash_uniform value
    df = spark.createDataFrame([
        Row(call_id="fixed_call_id_123", harness="claude_code", model_request="DeepSeek-V3.2"),
    ])
    result1 = compute_injection_probability(df).collect()[0].hash_uniform
    result2 = compute_injection_probability(df).collect()[0].hash_uniform
    assert result1 == result2